# Week 12 — Dataset Hygiene Report

**課程**：NS5116 電腦硬體與程式語言在行為科學實驗與大數據分析之應用  
**資料集**：`./data/messy_stroop_homework.csv`  
**Pipeline**：load → inspect → describe → fix → re-describe → analyse

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "./data/messy_stroop_homework.csv"

---
## A.0 Data Source & Schema Investigation

本節以**三個獨立來源交叉驗證**（triangulation）各欄位規則，每條規則皆標明依據來源。

---

### 來源 1 — 生成器(generator)逆向工程紀錄

閱讀 `./data/generate_messy_stroop_homework.py`後，有以下發現：

**固定 seed**：第 30 行 `np.random.seed(2026)` — 相同 seed 跑兩次會得到完全相同的資料。

**欄位生成方式**：
- `trial_id`：第 38 行，`np.arange(1, N + 1)` — 應為 1–240 的唯一整數 (先前設定 N = 240)。
- `subject_id`：第 39 行，從 `[101, 102, 103, 104, 105, 106]` 隨機抽取 — 應有 6 個 distinct 值。
- `condition`：第 40–41 行，從 `["congruent", "congruent ", "incongruent", "Incongruent"]` 抽取（含尾端空白與大小寫不一致）。
- `rt_ms`：第 42 行，`np.random.normal(520, 90, N)` — 期望之常模分布: 520 ms, SD 90 ms。Incongruent 條件另加 80 ms（第 54 行）。
- `accuracy`：第 43 行，從 `[0, 1, 1, 1, 1]` 抽取 — 應為 0/1 整數。
- `age`：第 44–45 行，從離散值 `[22, 26, 30, ..., 74, -1, 888, np.nan]` 抽取。

**刻意注入的 messy patterns 程式行**：

```python
# 問題 1：rt_ms 字串 sentinel — 第 62–68 行
rt_col.iloc[missing_idx] = "missing"   # 8 rows
rt_col.iloc[dash_idx]    = "--"        # 5 rows

# 問題 2：rt_ms 數值 sentinel — 第 76–82 行
rt_col.iloc[neg_idx] = -1    # 4 rows（負值 sentinel）
rt_col.iloc[pos_idx] = 9999  # 4 rows（正值 sentinel）

# 問題 3：condition 縮寫 — 第 89–91 行
df.loc[abbrev_idx[:3], "condition"] = "con"
df.loc[abbrev_idx[3:], "condition"] = "incong."

# 問題 4：age 的 sentinel — 第 44–45 行
# 生成時直接混入 -1 與 888 兩種 sentinel
"age": np.random.choice([22, 26, ..., 74, -1, 888, np.nan], N)

# 問題 5：accuracy 字串混入 — 第 96–101 行
acc_col.iloc[k] = "True" if i % 2 == 0 else "False"  # 10 rows

# 問題 6：重複 rows — 第 107–113 行
dup_indices = np.random.choice(N, 3, replace=False)
df = pd.concat([df, duplicates], ignore_index=True)  # 3 rows 重複
```

---

### 來源 2 — RT outliner exclusion 標準引用之參考文獻

**引用**：Lamers, M. J., Roelofs, A., & Rabeling-Keus, I. M. (2010). Selective attention and response set in the Stroop task. *Memory & Cognition*, *38*(7), 893–904.

從該文獻採用 Stroop task 的 RT outlier exclusion 標準：
- **排除下限**：RT < 200 ms
- **排除上限**：RT > 2000 ms
- **Stroop effect 量級**：文獻中 incongruent 與 congruent 條件差異典型落在約 61–93 ms。

**取捨說明**：根據 Lamers et al. (2010)，本作業採用的 RT outlier exclusion 標準為 200–2000 ms。使用嚴格範圍可排除非任務性反應(e.g., 亂按、不專注之試次)，同時正常 trial（generator: Mean = 520 ms, SD = 90 ms）幾乎不會被誤刪。

---

### 來源 3 — 資料本身（Data-Driven Discovery）

見 A.3 小節的 descriptive statistics 輸出。預覽發現：
- `rt_ms` dtype 為 `object`（字串）而非 `float` → 印證 generator 注入了字串 sentinel。
- `rt_ms` 觀察到 `"missing"` 與 `"--"` 兩種非數值字串 → 與 generator 第 62–68 行一致。
- `age` 出現 `-1` 與 `888` → 印證 generator 第 44–45 行的雙 sentinel。
- `condition` 出現 6 種 label（正確只應有 2 個: `congruent` 和 `incongruent`）→ 印證 generator 的 4 種 label + 縮寫注入。
- 「只從資料看不出來，但生成器告訴我的規則」：(1) `condition` 中有只在最後多了一個空格的sentinel  `congruent `，這是從資料檢閱是上很難直接看出來的；(2) 由於 ganerator 也是在特定範圍的資料點中隨機抽取 `age` 來填入每個 trial 的資料列，而不是根據每位受試者各自給予一固定年齡，所以會出現一位受試者有多種不同年齡的情況。這在現實世界不可能發生，但在本次作業中，我打算先根據老師給的 generator 設定來完成作業，先暫時無視這個問題。只要之後做圖時完全避開 `age` 這個變項即可。

---

### A.0 整合 Schema 表

| 欄位 | 預期型別 | 合理範圍 | 已知 sentinel | 依據 |
|------|----------|----------|---------------|------|
| `trial_id` | int | 1–240 | 無（但有重複 row） | generator 第 38、107–113 行 |
| `subject_id` | int | {101,102,103,104,105,106} | 無 | generator 第 39 行 |
| `condition` | category | {congruent, incongruent} | 尾端空白、大小寫、縮寫 | generator 第 40–41、89–91 行 |
| `rt_ms` | float | 200–2000 ms | `"missing"`, `"--"`, `-1`, `9999` | Lamers et al. (2010) + generator 第 62–82 行 + 觀察 |
| `accuracy` | int | {0, 1} | `"True"`, `"False"` | generator 第 96–101 行 |
| `age` | float | 22–74 | `-1`, `888`, `NaN` | generator 第 44–45 行 |

---
## A.1 Load

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

---
## A.2 Inspect

In [ ]:
print("shape:", df_raw.shape)
print()
print("dtypes:")
print(df_raw.dtypes)
print()
print("head(5):")
df_raw.head(5)

**第一眼觀察**：

1. 資料有 243 rows（預期 240 + 3 重複）× 6 columns。`rt_ms` 的 dtype 顯示為 `object` 而非數值，代表欄位中混有字串，無法直接用於計算。
2. `age` 欄有可見的 `-1` 與 `888`，明顯超出合理年齡範圍。
3. `condition` 欄可以看到`"Incongruent"` 首字母大寫，代表 label 方式未統一。

---
## A.3 Describe — 健康檢查

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all")

In [ ]:
print("=== Missing values ===")
print(df_raw.isnull().sum())

In [ ]:
print("=== value_counts for object columns ===")
for col in df_raw.select_dtypes("object"):
    print(f"\n--- {col} ---")
    print(df_raw[col].value_counts(dropna=False))

In [ ]:
# [Visualisation] Figure 1: rt_ms 分佈（需先嘗試轉數值）
rt_numeric = pd.to_numeric(df_raw["rt_ms"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(rt_numeric.dropna(), bins=40, edgecolor="white")
axes[0].axvline(200, color="red", linestyle="--", label="lower bound 200 ms")
axes[0].axvline(2000, color="orange", linestyle="--", label="upper bound 2000 ms")
axes[0].set_xlabel("rt_ms (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Figure 1. RT (ms) distribution (raw numeric values)")
axes[0].legend()

# Missing value bar chart
missing = df_raw.isnull().sum()
axes[1].bar(missing.index, missing.values, color="steelblue")
axes[1].set_title("Missing values per column")
axes[1].set_ylabel("Count")
axes[1].set_xlabel("Column")

plt.tight_layout()
plt.show()

**至少四個資料品質問題**：

**問題 1：`rt_ms` 為字串型別，含兩種字串 sentinel**  
- 觀察：`df.dtypes` 顯示 `rt_ms` dtype = `object`；`value_counts(dropna=False)` 看到 `"missing"` 出現 8 次、`"--"` 出現 5 次。  
- 原因：generator 第 62–68 行刻意將這些 index 的值替換為字串，導致整欄被 pandas 解讀為 object。

**問題 2：`rt_ms` 含數值 sentinel（-1 與 9999）**  
- 觀察：`describe()` 的 min 顯示出現負值；`value_counts()` 可見 `-1` 與 `9999`。這兩個值不在生理可能範圍（200–2000 ms）內。  
- 原因：generator 第 76–82 行注入 `-1`（表示「無反應」）與 `9999`（表示「超時」），是兩種不同語意的 sentinel。

**問題 3：`condition` 有 6 種 label，應只有 2 種**  
- 觀察：`value_counts()` 顯示 `"congruent"`、`"congruent "`（含尾端空白）、`"incongruent"`、`"Incongruent"`、`"con"`、`"incong."`。  
- 原因：generator 第 40–41 行原始抽樣已含不一致，第 89–91 行又注入縮寫，共造成 6 種變體。

**問題 4：`age` 含兩種數值 sentinel（-1 與 888）**  
- 觀察：`describe()` 的 min = -1.0、max = 888.0，皆超出合理年齡範圍（22–74）；`value_counts(dropna=False)` 確認兩種 sentinel 都存在。  
- 原因：generator 第 44–45 行在生成時直接混入 `-1` 與 `888` 兩種不同語意的特殊值（常見於不同研究人員用不同慣例標記缺值）。

**問題 5：`accuracy` 混有字串 `"True"` / `"False"`**  
- 觀察：`value_counts(dropna=False)` 看到 `"True"` 約 5 次、`"False"` 約 5 次，與整數 0/1 混存。  
- 原因：generator 第 96–101 行模擬部分受試者的記錄使用不同 logging 格式（Python 布林 → 字串化）。

**問題 6：重複 rows（trial_id 重複）**  
- 觀察：`df.shape` 顯示 243 rows（generator 設定 N=240 + 3 重複）；`trial_id.duplicated().sum()` = 3。  
- 原因：generator 第 107–113 行模擬 subject 資料合併失誤，隨機複製 3 個 row 並 concat。

---
## A.4 Fix — Observation-driven Cleaning

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pure function: clean the messy Stroop dataset.
    Input df is never modified (df.copy() at top).
    Each step documents: observation → action → cost.
    """
    df = df.copy()

    # ── Step 1: rt_ms 字串 → float ─────────────────────────────────────────
    # Observation (A.3 問題 1): rt_ms dtype=object，含 "missing"(8)、"--"(5)
    # Action: pd.to_numeric(errors="coerce") 將非數值字串轉為 NaN
    # Cost: 這些 row 的 rt_ms 資訊完全遺失；但字串本身無法用於分析，無其他選擇
    # Schema link: A.0 rt_ms sentinel "missing", "--"
    df["rt_ms"] = pd.to_numeric(df["rt_ms"], errors="coerce")
    print(f"Step 1 (to_numeric rt_ms): dtype changed → {df['rt_ms'].dtype}, "
          f"NaN introduced = {df['rt_ms'].isna().sum()}")

    # ── Step 2: rt_ms 數值 sentinel → NaN ──────────────────────────────────
    # Observation (A.3 問題 2): rt_ms 出現 -1（負值 sentinel）與 9999（正值 sentinel）
    # Action: replace({-1: NaN, 9999: NaN})
    # Cost: 共 8 rows 被標記為缺失（原本這些 trial 的 RT 資訊本就不可用）
    # Schema link: A.0 rt_ms sentinel -1, 9999
    df["rt_ms"] = df["rt_ms"].replace({-1: np.nan, 9999: np.nan})
    print(f"Step 2 (replace rt_ms sentinels -1/9999): NaN count = {df['rt_ms'].isna().sum()}")

    # ── Step 3: rt_ms 超出合理範圍 → NaN，再 dropna ────────────────────────
    # Observation (A.3 問題 2 + A.0): rt_ms 分佈中有超出 200–2000 ms 的值
    # Action: 將 between(200, 2000) 外的數值設為 NaN，再 dropna(subset=["rt_ms"])
    # Cost: 極端但可能合理的 trial 被捨棄（約 1–3 rows），換取符合文獻標準的乾淨分佈
    # Schema link: A.0 rt_ms 合理範圍 200–2000 ms（Lamers et al., 2010）
    df.loc[~df["rt_ms"].between(200, 2000) & df["rt_ms"].notna(), "rt_ms"] = np.nan
    before = len(df)
    df = df.dropna(subset=["rt_ms"])
    print(f"Step 3 (rt_ms range 200–2000 ms): {before} → {len(df)}  (lost {before - len(df)} rows)")

    # ── Step 4: condition 標準化 ────────────────────────────────────────────
    # Observation (A.3 問題 3): 6 種 label，應只有 2 種
    # Action: strip() 移除尾端空白 → lower() 統一大小寫 → replace 縮寫
    # Cost: 「con」與「incong.」被假設為縮寫對應；若有其他意義則會分類錯誤
    # Schema link: A.0 condition sentinel（尾端空白、大小寫、縮寫）
    df["condition"] = df["condition"].str.strip().str.lower()
    df["condition"] = df["condition"].replace({"con": "congruent", "incong.": "incongruent"})
    df["condition"] = df["condition"].astype("category")
    print(f"Step 4 (condition normalize): labels = {df['condition'].cat.categories.tolist()}")

    # ── Step 5: accuracy 字串 → int ─────────────────────────────────────────
    # Observation (A.3 問題 5): accuracy 含 "True"(~5)、"False"(~5) 與整數 0/1 混存
    # Action: replace "True"→1、"False"→0，再 pd.to_numeric → Int64
    # Cost: 假設字串 "True" 語意等同整數 1；若有其他語意則轉換錯誤
    # Schema link: A.0 accuracy sentinel "True", "False"
    df["accuracy"] = df["accuracy"].replace({"True": 1, "False": 0})
    df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce").astype("Int64")
    print(f"Step 5 (accuracy recode): unique values = {sorted(df['accuracy'].dropna().unique())}")

    # ── Step 6: age sentinel → NaN ──────────────────────────────────────────
    # Observation (A.3 問題 4): age 出現 -1 與 888，超出合理範圍 22–74
    # Action: replace({-1: NaN, 888: NaN})
    # Cost: 共約 15–20 rows 的 age 變為缺失；若後續要分析 age 需再處理
    # Schema link: A.0 age sentinel -1, 888
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = df["age"].replace({-1.0: np.nan, 888.0: np.nan})
    print(f"Step 6 (age sentinels -1/888): age NaN count = {df['age'].isna().sum()}")

    # ── Step 7: 移除重複 rows ────────────────────────────────────────────────
    # Observation (A.3 問題 6): 243 rows 但 N=240，trial_id 有 3 個重複
    # Action: drop_duplicates() 移除完全相同的 row
    # Cost: 若重複是因為不同 run 的合法 trial（非合併失誤），會誤刪；
    #        但 generator 確認這是合併失誤（第 107–113 行），因此安全
    # Schema link: A.0 trial_id 重複問題
    before = len(df)
    df = df.drop_duplicates()
    print(f"Step 7 (drop_duplicates): {before} → {len(df)}  (lost {before - len(df)} rows)")

    return df.reset_index(drop=True)


df_clean = clean(df_raw)
print(f"\nFinal shape: {df_clean.shape}")

---
## A.5 Re-describe — 驗證 Fix

In [ ]:
df_clean.describe(include="all")

In [ ]:
print("=== condition value_counts (after clean) ===")
print(df_clean["condition"].value_counts(dropna=False))
print()
print("=== accuracy value_counts (after clean) ===")
print(df_clean["accuracy"].value_counts(dropna=False))
print()
print("=== age range (after clean) ===")
print(df_clean["age"].describe())

**相比 A.3，清理後的改變**：

1. **`rt_ms`**：dtype 從 `object` 變為 `float64`；`"missing"`、`"--"` 消失；`-1` 與 `9999` 消失；min ≥ 200、max ≤ 2000，完全在合理範圍內。
2. **`condition`**：label 從 6 種縮減為 2 種（`congruent`、`incongruent`），尾端空白、大小寫不一致、縮寫全部消除。
3. **`accuracy`**：`"True"`、`"False"` 消失，只剩 `0` 與 `1`，dtype 為 `Int64`。
4. **`age`**：`-1.0` 與 `888.0` 消失；min = 22、max = 74，符合 generator 設定。
5. **row 數**：在移除重複的 row 後，243 → 約 218–225。

---
## A.6 Analyse — Stroop Effect

In [ ]:
def analyse(df: pd.DataFrame, *, outlier_sd: float = 3.0) -> pd.DataFrame:
    """
    Compute mean RT, SD, and Stroop effect per condition.

    Parameters
    ----------
    df : cleaned DataFrame from clean()
    outlier_sd : SD multiplier for within-analysis trimming (default 3.0)

    Why outlier_sd is here, not in clean():
        clean() handles data-quality problems (sentinels, dtype errors, range
        violations determined by the schema).  outlier_sd is an *analysis*
        decision: different research questions (e.g., robust mean vs. full
        distribution) may apply different thresholds to the same clean data.
        Putting it in clean() would bake a single analytical choice permanently
        into the data, preventing reproducibility comparisons.
    """
    rows = []
    for cond in ["congruent", "incongruent"]:
        sub = df[df["condition"] == cond]["rt_ms"].dropna()
        if outlier_sd:
            mu, sigma = sub.mean(), sub.std()
            sub = sub[(sub - mu).abs() <= outlier_sd * sigma]
        rows.append({"condition": cond, "n": len(sub),
                     "mean_rt": sub.mean(), "sd_rt": sub.std()})

    result = pd.DataFrame(rows).set_index("condition")
    stroop_effect = result.loc["incongruent", "mean_rt"] - result.loc["congruent", "mean_rt"]
    print(result.round(2))
    print(f"\nStroop effect = {stroop_effect:.2f} ms")
    return result


stats = analyse(df_clean)

**解讀**：Stroop effect（incongruent − congruent mean RT）應落在約 +80 ms，與 generator 設定一致（第 54 行 `+ 80`），也符合 Lamers et al. (2010) 報告的典型量級（50–100 ms）。

**為何 `outlier_sd` 放在 `analyse()` 而非 `clean()`**：`clean()` 處理的是資料品質問題（sentinel、dtype 錯誤、違反 schema 的值），這些問題的答案由資料來源（generator 規格、文獻定義的合理範圍）決定，與研究問題無關。`outlier_sd` 則是分析層面的決策：不同研究問題（例如要估計 robust mean 或完整分佈）可能對同一份乾淨資料套用不同閾值。若把 `outlier_sd` 放進 `clean()`，就是將某個分析選擇永久寫死在資料裡，後續無法在不重跑 pipeline 的情況下改變閾值，違反了「cleaning 與 analysis 邊界」的紀律。